# TextAnalysis
Entrada: `acciones_preparadas.csv` (8 partidos, una fila cada uno). Calcula un embedding por partido, similitud entre todos (H1) y proyección sobre un eje A–H (H2).

## TEXT-01: leer el corpus

In [ ]:
import pandas as pd

corpus = pd.read_csv('acciones_preparadas.csv')
assert len(corpus) == 8 and corpus['id_partido'].is_unique
corpus[['id_partido', 'partido', 'num_propuestas']]

## TEXT-02: modelo de embeddings
SBERT multilingüe preentrenado, sin ajustar. El límite de tokens sale del propio modelo.

In [ ]:
from sentence_transformers import SentenceTransformer
import nltk
nltk.download('punkt_tab', quiet=True)

embedding_model = SentenceTransformer("paraphrase-multilingual-mpnet-base-v2")
LIMITE = embedding_model.max_seq_length

def contar_tokens(texto):
    return len(embedding_model.tokenizer.encode(texto, add_special_tokens=True))

print('Límite de tokens del modelo:', LIMITE)

## TEXT-03: fragmentar y verificar
Las propuestas ya son cortas; se comprueba el límite en vez de asumirlo. Se verifica que fragmentar no pierda texto.

In [ ]:
from nltk.tokenize import sent_tokenize

def dividir_por_palabras(oracion, limite=LIMITE):
    palabras = oracion.split()
    fragmentos, actual = [], []
    for palabra in palabras:
        candidato = ' '.join(actual + [palabra])
        if contar_tokens(candidato) > limite and actual:
            fragmentos.append(' '.join(actual))
            actual = [palabra]
        else:
            actual.append(palabra)
    if actual:
        fragmentos.append(' '.join(actual))
    return fragmentos

def fragmentar(texto, id_partido, limite=LIMITE):
    oraciones = sent_tokenize(texto, language='spanish')
    fragmentos = []
    for oracion in oraciones:
        if contar_tokens(oracion) <= limite:
            fragmentos.append(oracion)
        else:
            fragmentos.extend(dividir_por_palabras(oracion, limite))
    reconstruido = ''.join(fragmentos).replace(' ', '')
    original = texto.replace(' ', '')
    assert reconstruido == original, f'{id_partido}: la fragmentación no preserva el texto.'
    return fragmentos

for _, fila in corpus.iterrows():
    frags = fragmentar(fila['texto'], fila['id_partido'])
    ok = 'OK' if len(frags) == fila['num_propuestas'] else 'REVISAR'
    print(fila['id_partido'], '->', len(frags), 'fragmentos', f"(num_propuestas={fila['num_propuestas']})", ok)

## TEXT-04: un embedding por partido
Promedio normalizado de los embeddings de sus propuestas. Cada propuesta pesa igual — por eso `num_propuestas` queda registrado.

In [ ]:
import numpy as np

def embedding_partido(texto, id_partido):
    fragmentos = fragmentar(texto, id_partido)
    vectores = embedding_model.encode(fragmentos, normalize_embeddings=True)
    promedio = vectores.mean(axis=0)
    return promedio / np.linalg.norm(promedio)

corpus['embedding'] = [embedding_partido(t, i) for t, i in zip(corpus['texto'], corpus['id_partido'])]
E = np.stack(corpus['embedding'].values)
print('Matriz de embeddings:', E.shape)

## TEXT-05: H1 — similitud coseno

In [ ]:
import matplotlib.pyplot as plt

ids = corpus['id_partido'].tolist()
S = E @ E.T

fig, ax = plt.subplots(figsize=(6,5))
im = ax.imshow(S, vmin=-1, vmax=1, cmap='RdBu_r')
ax.set_xticks(range(8)); ax.set_xticklabels(ids)
ax.set_yticks(range(8)); ax.set_yticklabels(ids)
for i in range(8):
    for j in range(8):
        ax.text(j, i, f'{S[i,j]:.2f}', ha='center', va='center', fontsize=8)
plt.colorbar(im); plt.title('Similitud coseno entre partidos'); plt.tight_layout(); plt.show()

Sin whitening: con 8 documentos la covarianza tiene rango ≤7, insuficiente para un espacio de cientos de dimensiones. Se reporta el coseno crudo, sin umbral universal.

## TEXT-06: pares más y menos similares
Revisar `num_propuestas` antes de interpretar: un partido con más propuestas puede parecer más 'genérico', no necesariamente más moderado.

In [ ]:
import itertools

pares = [(ids[i], ids[j], S[i,j], corpus['num_propuestas'][i], corpus['num_propuestas'][j])
          for i, j in itertools.combinations(range(8), 2)]
pares.sort(key=lambda x: -x[2])

print('Más similares:')
for p in pares[:3]:
    print(f'  {p[0]}({p[3]}) - {p[1]}({p[4]}): {p[2]:.3f}')
print('Menos similares:')
for p in pares[-3:]:
    print(f'  {p[0]}({p[3]}) - {p[1]}({p[4]}): {p[2]:.3f}')

## TEXT-07: H2 — eje A–H
A y H son anclas por decisión del investigador, no por el modelo. Eje de A (-1) a H (+1).

In [ ]:
a = E[ids.index('A')]
h = E[ids.index('H')]
v = h - a
m = (a + h) / 2

s = 2 * (E - m) @ v / (v @ v)
corpus['posicion_eje'] = s
corpus[['id_partido', 'partido', 'posicion_eje', 'num_propuestas']].sort_values('posicion_eje')

## TEXT-08: distancia al eje
Cuánto del contenido de cada partido no se explica por la dimensión estatal-mercado:

$$\text{distancia}_i = \left\| (d_i - m) - \frac{s_i}{2}\,v \right\|$$

In [ ]:
def distancia_al_eje(d, m, v, s_i):
    proyeccion = (s_i / 2) * v
    return np.linalg.norm((d - m) - proyeccion)

corpus['distancia_eje'] = [distancia_al_eje(E[i], m, v, s[i]) for i in range(8)]
corpus[['id_partido', 'posicion_eje', 'distancia_eje']].sort_values('posicion_eje')

## TEXT-09: visualizar H2
Que A y H queden en -1 y +1 es diseño, no evidencia. La evidencia es la dispersión de B-G.

In [ ]:
fig, ax = plt.subplots(figsize=(7,4))
orden = corpus.sort_values('posicion_eje')
colores = ['tab:red' if p in ('A','H') else 'tab:blue' for p in orden['id_partido']]
ax.scatter(orden['posicion_eje'], range(8), c=colores)
for y, (_, fila) in enumerate(orden.iterrows()):
    ax.text(fila['posicion_eje'], y, f"  {fila['id_partido']} ({fila['num_propuestas']} prop.)", va='center')
ax.axvline(0, color='gray', linestyle='--', linewidth=0.8)
ax.set_yticks([]); ax.set_xlabel('Estatal-redistributivo (-1)  <->  Liberal-mercado (+1)')
plt.tight_layout(); plt.show()

disp = corpus.loc[~corpus['id_partido'].isin(['A','H']), 'posicion_eje']
print('Rango de B-G en el eje:', round(disp.max() - disp.min(), 3))

## Qué no responde este análisis
El eje resume solo "Economía y papel del Estado", no toda la ideología. La similitud entre propuestas no explica la causa de la coincidencia (puede ser convergencia genuina o lenguaje estándar del género). `num_propuestas` fue decisión de cada partido, no un dato neutral: afecta el promedio de H1 y debe leerse junto a los resultados.